## Problem and data description

### Decision problem

A fleet of identical vehicles must leave one depot, serve every customer exactly once, and return to the depot. Each customer has a non-negative demand, and the total demand assigned to a vehicle cannot exceed its capacity. The objective is to minimize total travel distance.

This is representative of last-mile delivery, collection, maintenance, and bike-rebalancing operations. A good solution lowers distance, vehicle use, fuel consumption, and operating time while satisfying all service and capacity requirements.

In [1]:
import json
import math
import os
import platform
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import folium
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pulp
import psutil
import requests
from IPython.display import FileLink, display
from scipy.spatial.distance import cdist

# Use one baseline seed so the primary synthetic experiment is reproducible.
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
pd.set_option("display.max_columns", None)

# Record the execution environment because hardware and solver versions affect runtime comparisons.
print("Python experiment environment loaded successfully.")
runtime_environment = pd.DataFrame(
    {
        "Component": [
            "Python", "Operating system", "Machine", "Processor",
            "Logical CPU cores", "Memory (GB)", "PuLP", "HiGHS",
            "Folium", "Timing boundary",
        ],
        "Value": [
            platform.python_version(),
            platform.platform(),
            platform.machine(),
            platform.processor() or os.environ.get("PROCESSOR_IDENTIFIER", "Not reported"),
            psutil.cpu_count(logical=True),
            round(psutil.virtual_memory().total / (1024 ** 3), 2),
            pulp.__version__,
            getattr(__import__("highspy"), "__version__", "Available through PuLP"),
            folium.__version__,
            "Includes model/population construction and optimization, but excludes data generation and plotting",
        ],
    }
)
display(runtime_environment)


Python experiment environment loaded successfully.


,Component,Value
0,Python,3.13.2
1,Operating system,Windows-11-10.0.26300-SP0
2,Machine,AMD64
3,Processor,"Intel64 Family 6 Model 183 Stepping 1, Genuine..."
4,Logical CPU cores,28
5,Memory (GB),31.75
6,PuLP,4.0.0
7,HiGHS,Available through PuLP
8,Folium,0.20.0
9,Timing boundary,Includes model/population construction and opt...


### Controlled synthetic benchmark

The controlled benchmark places a depot at the centre of a 100 by 100 service region and generates customers with demands between 1 and 10 units. One coordinate unit is treated as one abstract distance unit. Euclidean distance represents travel in an unobstructed or approximately grid-like region. This abstraction is useful for controlled algorithm comparison but does not reproduce one-way streets, traffic, or road curvature.

Separate seeded random streams generate coordinates and demands, so the experiments are reproducible and smaller instances are prefixes of larger instances generated with the same seed. Synthetic data is used here to control problem size and isolate scalability effects. A later section uses a saved real Citi Bike snapshot and road-distance results to provide real-world validation.

In [2]:
# Validate every instance before optimization so invalid inputs cannot produce misleading results.
def validate_instance_data(instance):
    required = {
        "num_nodes", "num_customers", "num_vehicles",
        "vehicle_capacity", "dist_matrix", "demands",
    }
    missing = sorted(required.difference(instance))
    if missing:
        raise ValueError(f"Missing instance fields: {missing}")

    n = int(instance["num_customers"])
    node_count = int(instance["num_nodes"])
    vehicle_count = int(instance["num_vehicles"])
    capacity = float(instance["vehicle_capacity"])
    demands = np.asarray(instance["demands"], dtype=float)
    distances = np.asarray(instance["dist_matrix"], dtype=float)

    if n < 2 or node_count != n + 1:
        raise ValueError("An instance must contain a depot and at least two customers.")
    if vehicle_count <= 0 or capacity <= 0:
        raise ValueError("Vehicle count and capacity must be positive.")
    if demands.shape != (node_count,) or distances.shape != (node_count, node_count):
        raise ValueError("Demand vector or distance-matrix dimensions do not match the node count.")
    if not np.all(np.isfinite(demands)) or np.any(demands < 0) or demands[0] != 0:
        raise ValueError("Demands must be finite and non-negative, with zero demand at the depot.")
    if np.any(demands[1:] > capacity):
        raise ValueError("At least one customer demand exceeds vehicle capacity.")
    if demands.sum() > vehicle_count * capacity:
        raise ValueError("Total demand exceeds total fleet capacity.")
    if not np.all(np.isfinite(distances)) or np.any(distances < 0):
        raise ValueError("Distances must be finite and non-negative.")
    if not np.allclose(np.diag(distances), 0):
        raise ValueError("The distance-matrix diagonal must be zero.")
    return True


# Generate a controlled CVRP instance with independent seeded streams for coordinates and demands.
def generate_vrp_instance(
    num_customers=15,
    grid_size=100,
    max_demand=10,
    vehicle_capacity=30,
    num_vehicles=4,
    seed=42,
):
    coordinate_rng = np.random.default_rng(seed)
    demand_rng = np.random.default_rng(seed + 1)

    # Fix the depot at the centre and sample customers throughout the service region.
    depot_coords = np.array([[grid_size / 2, grid_size / 2]], dtype=float)
    customer_coords = coordinate_rng.uniform(0, grid_size, size=(num_customers, 2))
    coords = np.vstack([depot_coords, customer_coords])
    dist_matrix = cdist(coords, coords, metric="euclidean")
    demands = np.r_[0, demand_rng.integers(1, max_demand + 1, size=num_customers)]

    data = {
        "num_nodes": num_customers + 1,
        "num_customers": num_customers,
        "num_vehicles": num_vehicles,
        "vehicle_capacity": vehicle_capacity,
        "coords": coords,
        "dist_matrix": dist_matrix,
        "demands": demands,
        "distance_unit": "abstract distance units",
        "seed": seed,
    }
    validate_instance_data(data)
    return data


data = generate_vrp_instance(
    num_customers=10,
    vehicle_capacity=25,
    num_vehicles=3,
    seed=SEED,
)

instance_summary = pd.DataFrame(
    {
        "Metric": ["Customers", "Vehicles", "Capacity per vehicle", "Total demand", "Total fleet capacity", "Seed"],
        "Value": [
            data["num_customers"],
            data["num_vehicles"],
            data["vehicle_capacity"],
            int(data["demands"].sum()),
            data["num_vehicles"] * data["vehicle_capacity"],
            data["seed"],
        ],
    }
)
display(instance_summary)

# Present the generated inputs explicitly so the experiment can be audited and reproduced.
node_labels = ["Depot"] + [f"Customer_{customer}" for customer in range(1, data["num_customers"] + 1)]
synthetic_dataset = pd.DataFrame(
    {
        "Node_ID": range(data["num_nodes"]),
        "Node_Type": ["Depot"] + ["Customer"] * data["num_customers"],
        "X_coordinate": data["coords"][:, 0],
        "Y_coordinate": data["coords"][:, 1],
        "Demand": data["demands"].astype(int),
    },
    index=node_labels,
)
synthetic_dataset.index.name = "Node"
print("Complete controlled synthetic dataset")
display(synthetic_dataset.round({"X_coordinate": 3, "Y_coordinate": 3}))

distance_matrix_display = pd.DataFrame(
    data["dist_matrix"],
    index=node_labels,
    columns=node_labels,
).round(2)
distance_matrix_display.index.name = "From / To"
print(f"Complete Euclidean distance matrix ({data['distance_unit']})")
display(distance_matrix_display)


,Metric,Value
0,Customers,10
1,Vehicles,3
2,Capacity per vehicle,25
3,Total demand,49
4,Total fleet capacity,75
5,Seed,42


Complete controlled synthetic dataset


,Node_ID,Node_Type,X_coordinate,Y_coordinate,Demand
Node,,,,,
Depot,0,Depot,50.000,50.000,0
Customer_1,1,Customer,77.396,43.888,6
Customer_2,2,Customer,85.860,69.737,7
Customer_3,3,Customer,9.418,97.562,5
Customer_4,4,Customer,76.114,78.606,1
Customer_5,5,Customer,12.811,45.039,6
Customer_6,6,Customer,37.080,92.676,1
Customer_7,7,Customer,64.387,82.276,3
Customer_8,8,Customer,44.341,22.724,9


Complete Euclidean distance matrix (abstract distance units)


,Depot,Customer_1,Customer_2,Customer_3,Customer_4,Customer_5,Customer_6,Customer_7,Customer_8,Customer_9,Customer_10
From / To,,,,,,,,,,,
Depot,0.00,28.07,40.93,62.52,38.73,37.52,44.59,35.34,27.86,43.96,35.31
Customer_1,28.07,0.00,27.20,86.61,34.74,64.59,63.29,40.53,39.25,43.45,20.01
Customer_2,40.93,27.20,0.00,81.35,13.18,77.11,53.90,24.87,62.72,70.27,7.26
Customer_3,62.52,86.61,81.35,0.00,69.34,52.63,28.09,57.05,82.59,102.15,81.01
Customer_4,38.73,34.74,13.18,69.34,0.00,71.65,41.49,12.29,64.28,75.12,16.81
Customer_5,37.52,64.59,77.11,52.63,71.65,0.00,53.46,63.61,38.63,57.56,72.26
Customer_6,44.59,63.29,53.90,28.09,41.49,53.46,0.00,29.22,70.33,88.23,54.39
Customer_7,35.34,40.53,24.87,57.05,12.29,63.61,29.22,0.00,62.84,76.42,26.51
Customer_8,27.86,39.25,62.72,82.59,64.28,38.63,70.33,62.84,0.00,19.76,55.78
